# Walmart Sales Forecasting — Exploratory Data Analysis
**Walmart Store Sales Forecast Dataset** | Feb 2010 – Oct 2012 | 45 Stores

This notebook explores the aggregated weekly sales data used to train the forecasting models.
We examine the overall trend, seasonality, holiday effects, external economic features,
and feature correlations to understand what drives model performance.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
sns.set_theme(style='whitegrid')
%matplotlib inline

from data_preprocessing import load_walmart_data, add_features, FEATURE_COLS

weekly   = load_walmart_data()
featured = add_features(weekly)

print(f'Raw weekly rows  : {len(weekly)}')
print(f'Usable rows      : {len(featured)}  (after lag_52 warmup)')
print(f'Date range       : {weekly["Date"].min().date()} --> {weekly["Date"].max().date()}')
print(f'Avg weekly sales : ${weekly["total_sales"].mean():,.0f}')
weekly.head()

## 1. Overall Weekly Sales Trend
Total weekly sales aggregated across all 45 stores.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(weekly['Date'], weekly['total_sales'] / 1e6,
        color='#2b6cb0', linewidth=1.5, marker='o', markersize=2)
ax.set_title('Walmart Total Weekly Sales — All 45 Stores (Feb 2010 – Oct 2012)', fontsize=13)
ax.set_ylabel('Weekly Sales ($M)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:.0f}M'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 2. Seasonality — Sales by Week of Year
Average weekly sales by ISO week number, revealing the Thanksgiving (Week 47) spike.

In [ ]:
featured['week_of_year_raw'] = featured['Date'].dt.isocalendar().week.astype(int)
weekly_avg = featured.groupby('week_of_year_raw')['total_sales'].mean() / 1e6

fig, ax = plt.subplots(figsize=(14, 4))
colors = ['#e53e3e' if w in [6, 47, 52] else '#2b6cb0' for w in weekly_avg.index]
ax.bar(weekly_avg.index, weekly_avg.values, color=colors, width=0.8)
ax.set_title('Average Weekly Sales by Week of Year', fontsize=13)
ax.set_xlabel('ISO Week Number')
ax.set_ylabel('Avg Sales ($M)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:.0f}M'))
for week, label in [(6, 'Super Bowl'), (47, 'Thanksgiving'), (52, 'Christmas')]:
    ax.annotate(label, xy=(week, weekly_avg.get(week, 0)),
                xytext=(week, weekly_avg.get(week, 0) + 1.5),
                ha='center', fontsize=8, color='#e53e3e',
                arrowprops=dict(arrowstyle='->', color='#e53e3e', lw=1))
plt.tight_layout()
plt.show()

## 3. Holiday vs Non-Holiday Weeks
Comparing average sales on weeks flagged as holidays vs regular weeks.

In [ ]:
holiday_avg    = weekly[weekly['is_holiday'] == True]['total_sales'].mean() / 1e6
non_holiday_avg = weekly[weekly['is_holiday'] == False]['total_sales'].mean() / 1e6

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar comparison
axes[0].bar(['Non-Holiday', 'Holiday'], [non_holiday_avg, holiday_avg],
            color=['#4299e1', '#e53e3e'], width=0.5)
axes[0].set_title('Avg Weekly Sales: Holiday vs Non-Holiday')
axes[0].set_ylabel('Avg Sales ($M)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:.0f}M'))
for i, v in enumerate([non_holiday_avg, holiday_avg]):
    axes[0].text(i, v + 0.2, f'${v:.1f}M', ha='center', fontsize=10)

# Box plot distribution
holiday_sales     = weekly[weekly['is_holiday'] == True]['total_sales'] / 1e6
non_holiday_sales = weekly[weekly['is_holiday'] == False]['total_sales'] / 1e6
axes[1].boxplot([non_holiday_sales, holiday_sales], labels=['Non-Holiday', 'Holiday'],
                patch_artist=True,
                boxprops=dict(facecolor='#bee3f8'),
                medianprops=dict(color='#2b6cb0', linewidth=2))
axes[1].set_title('Sales Distribution: Holiday vs Non-Holiday')
axes[1].set_ylabel('Weekly Sales ($M)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:.0f}M'))

plt.tight_layout()
plt.show()
print(f'Holiday premium: +{(holiday_avg/non_holiday_avg - 1)*100:.1f}% vs non-holiday weeks')

## 4. Year-over-Year Comparison
How weekly sales in 2011 compare to the same weeks in 2012, revealing the lag_52 signal.

In [ ]:
weekly['year'] = weekly['Date'].dt.year
weekly['woy']  = weekly['Date'].dt.isocalendar().week.astype(int)

pivot = weekly.pivot_table(index='woy', columns='year', values='total_sales') / 1e6

fig, ax = plt.subplots(figsize=(14, 4))
for year, color in zip([2010, 2011, 2012], ['#a0aec0', '#4299e1', '#e53e3e']):
    if year in pivot.columns:
        pivot[year].dropna().plot(ax=ax, label=str(year), color=color,
                                  linewidth=1.8, marker='o', markersize=3)
ax.set_title('Year-over-Year Weekly Sales Comparison', fontsize=13)
ax.set_xlabel('ISO Week Number')
ax.set_ylabel('Weekly Sales ($M)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:.0f}M'))
ax.legend(title='Year')
plt.tight_layout()
plt.show()
print('Strong year-over-year correlation explains why lag_52 is the most important feature (50.3%).')

## 5. External Economic Features
Trends in temperature, fuel price, CPI, unemployment, and total markdown over time.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
axes = axes.flatten()

features_to_plot = [
    ('temperature',    'Temperature (°F)',      '#f6ad55'),
    ('fuel_price',     'Fuel Price ($/gallon)',  '#fc8181'),
    ('cpi',            'CPI',                   '#68d391'),
    ('unemployment',   'Unemployment Rate (%)', '#76e4f7'),
    ('total_markdown', 'Total Markdown ($M)',   '#b794f4'),
    ('total_sales',    'Total Sales ($M)',       '#4299e1'),
]

for ax, (col, title, color) in zip(axes, features_to_plot):
    vals = weekly[col] / 1e6 if col in ['total_markdown', 'total_sales'] else weekly[col]
    ax.plot(weekly['Date'], vals, color=color, linewidth=1.2)
    ax.set_title(title, fontsize=10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.suptitle('External Economic Features Over Time', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6. Feature Correlation with Total Sales
Pearson correlation of each engineered feature against the target variable.

In [ ]:
corr = (featured[FEATURE_COLS + ['total_sales']]
        .corr()['total_sales']
        .drop('total_sales')
        .sort_values())

fig, ax = plt.subplots(figsize=(8, 9))
colors = ['#fc8181' if v < 0 else '#4299e1' for v in corr]
corr.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Feature Correlation with Total Weekly Sales', fontsize=13)
ax.set_xlabel('Pearson Correlation')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()
print('Top positive correlates:', corr.nlargest(5).index.tolist())
print('Top negative correlates:', corr.nsmallest(3).index.tolist())

## 7. Summary Statistics

In [ ]:
print('=== Weekly Sales Summary ===')
print(weekly['total_sales'].describe()
      .apply(lambda x: f'${x:,.0f}').to_string())

print(f'\n=== Key Business Metrics ===')
q4 = weekly[weekly['Date'].dt.quarter == 4]['total_sales'].mean()
q1 = weekly[weekly['Date'].dt.quarter == 1]['total_sales'].mean()
print(f'Q4 avg (Oct-Dec)      : ${q4:,.0f}')
print(f'Q1 avg (Jan-Mar)      : ${q1:,.0f}')
print(f'Q4 premium over Q1    : +{(q4/q1 - 1)*100:.0f}%')

woy = weekly.copy()
woy['woy'] = woy['Date'].dt.isocalendar().week.astype(int)
thanksgiving = woy[woy['woy'] == 47]['total_sales'].mean()
print(f'Thanksgiving week avg : ${thanksgiving:,.0f}')
print(f'Thanksgiving premium  : +{(thanksgiving/weekly["total_sales"].mean() - 1)*100:.0f}% vs annual avg')

print(f'\n=== Feature Engineering Summary ===')
print(f'Total features        : {len(FEATURE_COLS)}')
print(f'Usable training rows  : {len(featured)}  (raw: {len(weekly)}, warmup: {len(weekly)-len(featured)} weeks)')